In [3]:
"""
=============================================================================
NOTEBOOK 01: BUILD STOCK-LEVEL PANEL
=============================================================================
Proyecto: Detección de Insider Trading en el Congreso de EE.UU.
Maestría en Economía - UdeSA

Este notebook construye un panel a nivel (ACCIÓN, MES) en lugar de agregar
todo a nivel mercado. Esto aumenta las observaciones de ~150 a ~50,000+.

DIFERENCIA CLAVE vs. versión anterior:
- Antes: 1 observación por mes (S&P 500 agregado) = 150 obs
- Ahora: N acciones × T meses = ~15,000-25,000 obs

OUTPUT: panel_stock_month.parquet
=============================================================================
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm
import os

# =============================================================================
# 0. CONFIGURACIÓN
# =============================================================================

# Cambiar al directorio del proyecto
os.chdir('C:/Users/sebib/Documents/GitHub/US_Congress')
print(f"Directorio de trabajo: {os.getcwd()}")

# Paths
INPUT_TRADES = 'data/outputs/congress_trades_with_committees.parquet'
OUTPUT_DIR = 'data/prediction_bases'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Período de análisis
START_DATE = '2012-01-01'
END_DATE = '2024-12-31'

# Comités con información privilegiada potencial
INFO_COMMITTEES = [
    'Armed Services', 'Financial Services', 'Energy and Commerce',
    'Intelligence', 'Select Committee on Intelligence',
    'Ways and Means', 'Appropriations', 'Health Education Labor and Pensions',
    'Banking, Housing and Urban Affairs', 'Finance',
    'Judiciary', 'Commerce, Science and Transportation'
]

print("="*70)
print("NOTEBOOK 01: BUILD STOCK-LEVEL PANEL")
print("="*70)

# =============================================================================
# 1. CARGAR DATOS
# =============================================================================
print("\n[1] CARGANDO DATOS...")

df = pd.read_parquet(INPUT_TRADES)
print(f"    Trades cargados: {len(df):,}")
print(f"    Columnas: {len(df.columns)}")

# Ver las columnas disponibles
print(f"\n    Columnas disponibles:")
for i, col in enumerate(df.columns):
    print(f"      {i+1:2}. {col}")

# =============================================================================
# 2. EXPLORAR DATOS
# =============================================================================
print("\n[2] EXPLORANDO DATOS...")

# Ver primeras filas
print("\n    Primeras filas:")
print(df.head(3).T)

# Ver tipos de datos
print("\n    Tipos de datos:")
print(df.dtypes)

# =============================================================================
# 3. LIMPIEZA BÁSICA Y FECHAS
# =============================================================================
print("\n[3] LIMPIEZA Y PREPARACIÓN DE FECHAS...")

# Identificar columna de fecha de trade
date_cols = [col for col in df.columns if 'date' in col.lower() or 'traded' in col.lower()]
print(f"    Columnas de fecha encontradas: {date_cols}")

# Buscar columna de fecha
if 'Traded' in df.columns:
    df['trade_date'] = pd.to_datetime(df['Traded'])
elif 'trade_date' in df.columns:
    df['trade_date'] = pd.to_datetime(df['trade_date'])
else:
    # Buscar cualquier columna que parezca fecha
    for col in df.columns:
        if 'date' in col.lower():
            df['trade_date'] = pd.to_datetime(df[col])
            print(f"    Usando columna '{col}' como fecha de trade")
            break

# Crear trade_month
df['trade_month'] = df['trade_date'].dt.to_period('M')
df['trade_year'] = df['trade_date'].dt.year

print(f"    Período: {df['trade_date'].min()} a {df['trade_date'].max()}")

# Identificar columna de ticker
ticker_cols = [col for col in df.columns if 'ticker' in col.lower()]
print(f"    Columnas de ticker encontradas: {ticker_cols}")

# Usar la columna de ticker correcta
if 'Ticker_Clean' in df.columns:
    ticker_col = 'Ticker_Clean'
elif 'ticker_clean' in df.columns:
    ticker_col = 'ticker_clean'
elif 'Ticker' in df.columns:
    ticker_col = 'Ticker'
    df['Ticker_Clean'] = df['Ticker'].str.upper().str.strip()
    ticker_col = 'Ticker_Clean'
else:
    ticker_col = ticker_cols[0] if ticker_cols else None
    
print(f"    Usando columna '{ticker_col}' como ticker")

# Filtrar período
df = df[(df['trade_date'] >= START_DATE) & (df['trade_date'] <= END_DATE)]
print(f"    Trades en período: {len(df):,}")
print(f"    Acciones únicas: {df[ticker_col].nunique():,}")

# =============================================================================
# 4. CREAR FEATURES A NIVEL TRADE
# =============================================================================
print("\n[4] CREANDO FEATURES A NIVEL TRADE...")

# --- 4.1 Dirección del trade ---
# Buscar columna de transacción
trans_col = None
for col in ['Transaction', 'transaction', 'Type', 'type']:
    if col in df.columns:
        trans_col = col
        break

if trans_col:
    df['is_buy'] = df[trans_col].str.lower().str.contains('purchase|buy', na=False).astype(int)
    df['is_sell'] = df[trans_col].str.lower().str.contains('sale|sell', na=False).astype(int)
else:
    print("    ⚠️ No se encontró columna de transacción, buscando alternativas...")
    # Intentar con otras columnas
    if 'is_buy' in df.columns:
        df['is_buy'] = df['is_buy'].astype(int)
        df['is_sell'] = 1 - df['is_buy']

df['trade_direction'] = df['is_buy'] - df['is_sell']
print(f"    Compras: {df['is_buy'].sum():,}, Ventas: {df['is_sell'].sum():,}")

# --- 4.2 Monto del trade ---
# Buscar columna de monto
amount_cols = [col for col in df.columns if 'amount' in col.lower() or 'size' in col.lower() or 'usd' in col.lower()]
print(f"    Columnas de monto encontradas: {amount_cols}")

if 'Trade_Size_USD' in df.columns:
    size_map = {
        '$1,001 - $15,000': 8000,
        '$15,001 - $50,000': 32500,
        '$50,001 - $100,000': 75000,
        '$100,001 - $250,000': 175000,
        '$250,001 - $500,000': 375000,
        '$500,001 - $1,000,000': 750000,
        '$1,000,001 - $5,000,000': 3000000,
        'Over $5,000,000': 7500000,
    }
    df['amount_proxy'] = df['Trade_Size_USD'].map(size_map).fillna(8000)
elif 'amount_proxy' in df.columns:
    pass  # Ya existe
else:
    df['amount_proxy'] = 8000  # Default

df['is_large_trade'] = (df['amount_proxy'] >= 100000).astype(int)

# --- 4.3 Timing ---
# Buscar columna de fecha de filing
if 'Filed' in df.columns:
    df['filed_date'] = pd.to_datetime(df['Filed'])
elif 'filed_date' in df.columns:
    df['filed_date'] = pd.to_datetime(df['filed_date'])
else:
    df['filed_date'] = df['trade_date']  # Default

df['disclosure_delay'] = (df['filed_date'] - df['trade_date']).dt.days
df['disclosure_delay'] = df['disclosure_delay'].clip(lower=0, upper=365)
df['long_delay'] = (df['disclosure_delay'] > 30).astype(int)

df['day_of_month'] = df['trade_date'].dt.day
df['days_in_month'] = df['trade_date'].dt.daysinmonth
df['end_of_month'] = (df['days_in_month'] - df['day_of_month'] <= 5).astype(int)

df['day_of_week'] = df['trade_date'].dt.dayofweek
df['is_monday'] = (df['day_of_week'] == 0).astype(int)
df['is_friday'] = (df['day_of_week'] == 4).astype(int)

print(f"    Disclosure delay promedio: {df['disclosure_delay'].mean():.1f} días")

# --- 4.4 Comités ---
def is_info_committee(committee_name):
    if pd.isna(committee_name):
        return 0
    for ic in INFO_COMMITTEES:
        if ic.lower() in str(committee_name).lower():
            return 1
    return 0

# Buscar columna de comité
comm_col = None
for col in df.columns:
    if 'committee' in col.lower():
        comm_col = col
        break

if comm_col:
    df['is_info_committee'] = df[comm_col].apply(is_info_committee)
    print(f"    Trades de comités informativos: {df['is_info_committee'].mean()*100:.1f}%")
else:
    df['is_info_committee'] = 0
    print("    ⚠️ No se encontró columna de comité")

# Chair/Ranking member
chair_cols = [col for col in df.columns if 'chair' in col.lower() or 'role' in col.lower()]
if chair_cols:
    df['is_chair'] = df[chair_cols[0]].fillna('').str.lower().str.contains('chair|ranking', na=False).astype(int)
else:
    df['is_chair'] = 0

# --- 4.5 Poder del político ---
# Antigüedad
seniority_cols = [col for col in df.columns if 'year' in col.lower() and 'position' in col.lower()]
if seniority_cols:
    df['years_in_position'] = pd.to_numeric(df[seniority_cols[0]], errors='coerce').fillna(0)
elif 'seniority_years' in df.columns:
    df['years_in_position'] = pd.to_numeric(df['seniority_years'], errors='coerce').fillna(0)
else:
    df['years_in_position'] = 0

df['is_senior'] = (df['years_in_position'] >= 10).astype(int)

# Net worth
nw_cols = [col for col in df.columns if 'worth' in col.lower() or 'net_worth' in col.lower()]
if nw_cols:
    df['net_worth'] = pd.to_numeric(df[nw_cols[0]], errors='coerce').fillna(0)
    median_nw = df.loc[df['net_worth'] > 0, 'net_worth'].median() if (df['net_worth'] > 0).any() else 0
    df['is_wealthy'] = (df['net_worth'] > median_nw).astype(int)
else:
    df['net_worth'] = 0
    df['is_wealthy'] = 0

# Senador
chamber_cols = [col for col in df.columns if 'chamber' in col.lower()]
if chamber_cols:
    df['is_senator'] = df[chamber_cols[0]].astype(str).str.lower().str.contains('senate|2').astype(int)
else:
    df['is_senator'] = 0

# Power index
df['power_index'] = df['is_chair'] + df['is_senator'] + df['is_senior'] + df['is_info_committee']
print(f"    Power index promedio: {df['power_index'].mean():.2f}")

# --- 4.6 Partido ---
party_cols = [col for col in df.columns if 'party' in col.lower()]
if party_cols:
    party_col = party_cols[0]
    df['is_democrat'] = df[party_col].astype(str).str.upper().str.contains('D|DEM|1').astype(int)
    df['is_republican'] = df[party_col].astype(str).str.upper().str.contains('R|REP|2').astype(int)
else:
    df['is_democrat'] = 0
    df['is_republican'] = 0

# --- 4.7 Comportamiento ---
# Buscar columna de nombre
name_cols = [col for col in df.columns if col.lower() in ['name', 'politician', 'representative']]
if not name_cols:
    name_cols = [col for col in df.columns if 'name' in col.lower()]
name_col = name_cols[0] if name_cols else df.columns[0]

# Frequent trader
trader_counts = df.groupby(name_col)['trade_date'].count()
frequent_threshold = trader_counts.quantile(0.75)
frequent_traders = trader_counts[trader_counts >= frequent_threshold].index
df['frequent_trader'] = df[name_col].isin(frequent_traders).astype(int)

# First time trading this stock
df = df.sort_values([name_col, ticker_col, 'trade_date'])
df['first_time'] = (~df.duplicated(subset=[name_col, ticker_col], keep='first')).astype(int)

# Direction change
df['prev_is_buy'] = df.groupby([name_col, ticker_col])['is_buy'].shift(1)
df['direction_change'] = ((df['is_buy'] != df['prev_is_buy']) & df['prev_is_buy'].notna()).astype(int)

print(f"    First time trades: {df['first_time'].mean()*100:.1f}%")

# --- 4.8 Coordinación ---
print("    Calculando coordinación...")

# Múltiples políticos misma acción mismo día
daily_traders = df.groupby(['trade_date', ticker_col])[name_col].nunique().reset_index()
daily_traders.columns = ['trade_date', ticker_col, 'n_traders_same_day']
df = df.merge(daily_traders, on=['trade_date', ticker_col], how='left')
df['coordinated'] = (df['n_traders_same_day'] >= 2).astype(int)

# Mismo partido
if party_cols:
    party_traders = df.groupby(['trade_date', ticker_col, party_cols[0]])[name_col].nunique().reset_index()
    party_traders.columns = ['trade_date', ticker_col, party_cols[0], 'n_party_traders']
    df = df.merge(party_traders, on=['trade_date', ticker_col, party_cols[0]], how='left')
    df['party_coordinated'] = (df['n_party_traders'] >= 2).astype(int)
else:
    df['party_coordinated'] = 0

print(f"    Trades coordinados: {df['coordinated'].mean()*100:.1f}%")

# =============================================================================
# 5. COLAPSAR A NIVEL (ACCIÓN, MES)
# =============================================================================
print("\n[5] COLAPSANDO A NIVEL (ACCIÓN, MES)...")

# Primero calcular políticos únicos
unique_politicians = df.groupby([ticker_col, 'trade_month'])[name_col].nunique().reset_index()
unique_politicians.columns = ['ticker', 'month', 'cong_unique_politicians']

# Aggregar
agg = df.groupby([ticker_col, 'trade_month']).agg(
    # === CONTEOS BÁSICOS ===
    cong_total_trades=('is_buy', 'count'),
    cong_buy_count=('is_buy', 'sum'),
    cong_sell_count=('is_sell', 'sum'),
    
    # === MONTOS ===
    cong_total_amount=('amount_proxy', 'sum'),
    cong_large_trades=('is_large_trade', 'sum'),
    
    # === TIMING ===
    cong_avg_disclosure_delay=('disclosure_delay', 'mean'),
    cong_long_delay_trades=('long_delay', 'sum'),
    cong_end_of_month_trades=('end_of_month', 'sum'),
    cong_monday_trades=('is_monday', 'sum'),
    cong_friday_trades=('is_friday', 'sum'),
    
    # === COMITÉS ===
    cong_info_committee_trades=('is_info_committee', 'sum'),
    cong_chair_trades=('is_chair', 'sum'),
    
    # === PODER ===
    cong_senior_trades=('is_senior', 'sum'),
    cong_wealthy_trades=('is_wealthy', 'sum'),
    cong_senator_trades=('is_senator', 'sum'),
    cong_avg_power_index=('power_index', 'mean'),
    cong_max_power_index=('power_index', 'max'),
    cong_avg_seniority=('years_in_position', 'mean'),
    
    # === PARTIDO ===
    cong_dem_trades=('is_democrat', 'sum'),
    cong_rep_trades=('is_republican', 'sum'),
    
    # === COMPORTAMIENTO ===
    cong_frequent_trader_trades=('frequent_trader', 'sum'),
    cong_first_time_trades=('first_time', 'sum'),
    cong_direction_change_trades=('direction_change', 'sum'),
    
    # === COORDINACIÓN ===
    cong_coordinated_trades=('coordinated', 'sum'),
    cong_party_coordinated_trades=('party_coordinated', 'sum'),
    cong_max_traders_same_day=('n_traders_same_day', 'max'),
    
).reset_index()

agg.columns = ['ticker', 'month'] + list(agg.columns[2:])

# Merge unique politicians
agg = agg.merge(unique_politicians, on=['ticker', 'month'], how='left')

# === VARIABLES DERIVADAS ===
print("    Creando variables derivadas...")

# Señal neta
agg['cong_net'] = agg['cong_buy_count'] - agg['cong_sell_count']
agg['cong_buy_ratio'] = agg['cong_buy_count'] / agg['cong_total_trades']

# CSI (Congressional Sentiment Index)
agg['cong_csi'] = agg['cong_net'] / agg['cong_total_trades']

# Ratios sobre total
total = agg['cong_total_trades']
agg['cong_info_ratio'] = agg['cong_info_committee_trades'] / total
agg['cong_chair_ratio'] = agg['cong_chair_trades'] / total
agg['cong_senior_ratio'] = agg['cong_senior_trades'] / total
agg['cong_senator_ratio'] = agg['cong_senator_trades'] / total
agg['cong_dem_ratio'] = agg['cong_dem_trades'] / total
agg['cong_coordinated_ratio'] = agg['cong_coordinated_trades'] / total
agg['cong_first_time_ratio'] = agg['cong_first_time_trades'] / total
agg['cong_large_ratio'] = agg['cong_large_trades'] / total
agg['cong_long_delay_ratio'] = agg['cong_long_delay_trades'] / total

# Intensidad
agg['cong_intensity'] = agg['cong_total_trades'] / agg['cong_unique_politicians']

# Señales binarias
agg['cong_consensus_buy'] = (agg['cong_buy_ratio'] > 0.7).astype(int)
agg['cong_consensus_sell'] = (agg['cong_buy_ratio'] < 0.3).astype(int)
agg['cong_multiple_politicians'] = (agg['cong_unique_politicians'] > 1).astype(int)

# Señales compuestas
agg['cong_smart_money'] = ((agg['cong_net'] > 0) & 
                           (agg['cong_info_committee_trades'] > 0) & 
                           (agg['cong_chair_trades'] > 0)).astype(int)

agg['cong_strong_buy'] = ((agg['cong_csi'] > 0.5) & 
                          (agg['cong_unique_politicians'] >= 2)).astype(int)

agg['cong_strong_sell'] = ((agg['cong_csi'] < -0.5) & 
                           (agg['cong_unique_politicians'] >= 2)).astype(int)

panel_cong = agg.copy()

print(f"    Panel generado: {len(panel_cong):,} observaciones (ticker-month)")
print(f"    Acciones: {panel_cong['ticker'].nunique():,}")
print(f"    Meses: {panel_cong['month'].nunique()}")

# =============================================================================
# 6. GUARDAR PANEL
# =============================================================================
print("\n[6] GUARDANDO...")

output_path = os.path.join(OUTPUT_DIR, 'panel_congress_stock_month.parquet')
panel_cong.to_parquet(output_path, index=False)
print(f"    Guardado: {output_path}")

csv_path = os.path.join(OUTPUT_DIR, 'panel_congress_stock_month.csv')
panel_cong.to_csv(csv_path, index=False)
print(f"    Guardado: {csv_path}")

# =============================================================================
# 7. RESUMEN
# =============================================================================
print("\n" + "="*70)
print("RESUMEN - PANEL A NIVEL ACCIÓN-MES")
print("="*70)

cong_vars = [c for c in panel_cong.columns if c.startswith('cong_')]

print(f"""
DIMENSIONES:
  Observaciones:     {len(panel_cong):,}
  Acciones únicas:   {panel_cong['ticker'].nunique():,}
  Meses:             {panel_cong['month'].nunique()}
  Variables:         {len(panel_cong.columns)}

VARIABLES DE CONGRESO ({len(cong_vars)}):
  Conteos:           cong_total_trades, cong_buy_count, cong_sell_count
  Señales:           cong_net, cong_csi, cong_buy_ratio
  Comités:           cong_info_ratio, cong_chair_ratio
  Poder:             cong_senior_ratio, cong_senator_ratio, cong_avg_power_index
  Coordinación:      cong_coordinated_ratio, cong_multiple_politicians
  Compuestas:        cong_smart_money, cong_strong_buy, cong_strong_sell

COMPARACIÓN CON ENFOQUE ANTERIOR:
  Antes (S&P 500):   ~150 observaciones
  Ahora (acciones):  ~{len(panel_cong):,} observaciones

PRÓXIMO PASO:
  Notebook 02: Agregar datos de mercado y calcular retornos futuros
""")

print("="*70)
print("✅ NOTEBOOK 01 COMPLETADO")
print("="*70)

# =============================================================================
# 8. VISTA PREVIA DEL PANEL
# =============================================================================
print("\n📊 VISTA PREVIA DEL PANEL:")
print(panel_cong.head(10))

print("\n📊 ESTADÍSTICAS DE VARIABLES CLAVE:")
key_vars = ['cong_total_trades', 'cong_net', 'cong_csi', 'cong_unique_politicians', 
            'cong_info_ratio', 'cong_avg_power_index']

print(panel_cong[key_vars].describe().round(3))

Directorio de trabajo: C:\Users\sebib\Documents\GitHub\US_Congress
NOTEBOOK 01: BUILD STOCK-LEVEL PANEL

[1] CARGANDO DATOS...
    Trades cargados: 99,609
    Columnas: 97

    Columnas disponibles:
       1. Ticker
       2. TickerType
       3. Company
       4. Traded
       5. Transaction
       6. Trade_Size_USD
       7. Status
       8. Subholding
       9. Description
      10. Name
      11. BioGuideID
      12. Filed
      13. Party
      14. District
      15. Chamber
      16. Comments
      17. Quiver_Upload_Time
      18. excess_return
      19. State
      20. last_modified
      21. Ticker_Clean
      22. is_equity
      23. trade_id
      24. return_t
      25. abs_return_t
      26. return_overnight
      27. return_intraday
      28. momentum_5d
      29. momentum_20d
      30. momentum_60d
      31. momentum_252d
      32. realized_vol_30d
      33. parkinson_vol_30d
      34. realized_vol_60d
      35. vol_of_vol_60d
      36. realized_vol_252d
      37. volume_t
 